### Calculation of EarthQuake's Impact Damage Potential

### Load The Dataset

In [2]:
import pandas as pd

df = pd.read_csv('C:/Users/SRIVIDYA GAJJALA/Datasets/preprocessed_earthquake_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109980 entries, 0 to 109979
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   latitude   109980 non-null  float64
 1   longitude  109980 non-null  float64
 2   depth      109980 non-null  float64
 3   mag        109980 non-null  float64
 4   magType    109980 non-null  object 
 5   rms        109980 non-null  float64
 6   type       109980 non-null  object 
 7   status     109980 non-null  object 
 8   Year       109980 non-null  int64  
 9   Month      109980 non-null  int64  
 10  Day        109980 non-null  int64  
 11  Hour       109980 non-null  int64  
 12  Minute     109980 non-null  int64  
 13  Second     109980 non-null  int64  
 14  DayOfWeek  109980 non-null  int64  
dtypes: float64(5), int64(7), object(3)
memory usage: 12.6+ MB


### 1. Magnitude Type Prioritization and Standardization

In [3]:
def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5 # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    # Add more conversions as needed
    elif mag_type.lower() == 'ml':
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type
        return np.nan

In [4]:
# Apply the conversion to the DataFrame
import numpy as np
df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
df[['mag', 'magType', 'Mw']].head(10)

,mag,magType,Mw
0,3.296890,mw,3.296890
1,2.921017,mw,2.921017
2,4.716854,mw,4.716854
3,4.299218,mw,4.299218
4,3.443063,mw,3.443063
5,3.860699,mw,3.860699
6,2.670435,mw,2.670435
7,2.190153,mw,2.190153
8,2.712198,mw,2.712198
9,2.231916,mw,2.231916


#### 2. Calculate Damage Potential Using HAZUS-Style Formula

In [5]:
def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8 # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    # Return the final damage potential score
    return damage_potential

In [6]:
# Apply the conversion to the DataFrame
df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
df[['Mw', 'depth', 'damage_potential']].head(10)

,Mw,depth,damage_potential
0,3.296890,-0.430428,10.000000
1,2.921017,-0.430428,10.000000
2,4.716854,-0.291053,10.000000
3,4.299218,-0.430428,10.000000
4,3.443063,-0.430428,10.000000
5,3.860699,-0.476886,10.000000
6,2.670435,-0.476886,10.000000
7,2.190153,-0.430428,9.574951
8,2.712198,-0.383970,10.000000
9,2.231916,-0.430428,9.679213


Observations:

*feature engineering is the process of creating,transforming and selecting data feautures to improve machien learning model's performance.

*we converted different mag types into 1type to make it easy for model to find trends and mw(moment magintude)is the standard way which is considered most accurate and consistent.

*now from existing magnitude values and depth we are creating a new feature called damage potential using hazus which provides emperical formula that estimates peak ground accelaration- how much the ground shakes.